In [1]:
import runpod
import boto3

/home/jorge/miniconda3/envs/ml-platform/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import json
import os

# Configura tu API Key (puedes obtenerla en la configuración de tu cuenta RunPod)
runpod.api_key = ""

def obtener_precios_gpu(gpu_name):
    try:
        # Obtiene detalles técnicos y de mercado para la GPU especificada
        gpu_info = runpod.get_gpu(gpu_name)
        
        print(f"--- Precios para {gpu_info['displayName']} ---")
        print(f"Precio Community Cloud: ${gpu_info['communityPrice']}/hr")
        print(f"Precio Secure Cloud: ${gpu_info['securePrice']}/hr")

        print(json.dumps(gpu_info, indent=2))

            
    except Exception as e:
        print(f"Error al obtener datos: {e}")

# Ejemplo de uso
obtener_precios_gpu("NVIDIA RTX 4090")

Error al obtener datos: No GPU found with the specified ID, run runpod.get_gpus() to get a list of all GPUs


In [ ]:
lista_gpus_low = [
    "NVIDIA A40",
    "NVIDIA GeForce RTX 4090",
    "NVIDIA GeForce RTX 5090",
    "NVIDIA L4",
    "NVIDIA RTX PRO 4500 Blackwell"
]

lista_gpus_medium = [
    "NVIDIA L40S",
    "NVIDIA RTX 6000 Ada Generation",
    "NVIDIA RTX PRO 6000 Blackwell Server Edition",
    "NVIDIA L40"
]

lista_gpus_high = [
    "NVIDIA H100 80GB HBM3",   # H100 SXM
    "NVIDIA H100 PCIe",
    "NVIDIA H100 NVL",
    "NVIDIA H200 NVL"
]

In [3]:
all_gpus = runpod.get_gpus()

In [4]:
obtener_precios_gpu("NVIDIA RTX 2000 Ada Generation")

--- Precios para RTX 2000 Ada ---
Precio Community Cloud: $0.5/hr
Precio Secure Cloud: $0.24/hr
{
  "maxGpuCount": 8,
  "id": "NVIDIA RTX 2000 Ada Generation",
  "displayName": "RTX 2000 Ada",
  "manufacturer": "Nvidia",
  "memoryInGb": 16,
  "cudaCores": 0,
  "secureCloud": true,
  "communityCloud": false,
  "securePrice": 0.24,
  "communityPrice": 0.5,
  "oneMonthPrice": null,
  "threeMonthPrice": 0.196,
  "oneWeekPrice": null,
  "communitySpotPrice": null,
  "secureSpotPrice": 0.14,
  "lowestPrice": {
    "minimumBidPrice": null,
    "uninterruptablePrice": 0.5
  }
}


In [1]:
import subprocess
import json

result = subprocess.run(["sky", "show-gpus", "RTX2000-Ada"], capture_output=True, text=True)
print(result.stdout)

GPU          QTY  CLOUD   INSTANCE_TYPE          DEVICE_MEM  vCPUs  HOST_MEM  HOURLY_PRICE  HOURLY_SPOT_PRICE  REGION  
RTX2000-Ada  1.0  RunPod  1x_RTX2000-Ada_SECURE  0GB         6      31GB      $ 0.240       $ 0.140            CA      
RTX2000-Ada  2.0  RunPod  2x_RTX2000-Ada_SECURE  0GB         12     62GB      $ 0.480       $ 0.280            CA      
RTX2000-Ada  3.0  RunPod  3x_RTX2000-Ada_SECURE  0GB         18     93GB      $ 0.720       $ 0.420            CA      
RTX2000-Ada  4.0  RunPod  4x_RTX2000-Ada_SECURE  0GB         24     124GB     $ 0.960       $ 0.560            CA      
RTX2000-Ada  5.0  RunPod  5x_RTX2000-Ada_SECURE  0GB         30     155GB     $ 1.200       $ 0.700            CA      
RTX2000-Ada  6.0  RunPod  6x_RTX2000-Ada_SECURE  0GB         36     186GB     $ 1.440       $ 0.840            CA      
RTX2000-Ada  7.0  RunPod  7x_RTX2000-Ada_SECURE  0GB         42     217GB     $ 1.680       $ 0.980            CA      
RTX2000-Ada  8.0  RunPod  8x_RTX2000-Ada

In [7]:
import pandas as pd
import re

data = result.stdout

# Regex para capturar columnas correctamente
pattern = re.compile(
    r"(\S+)\s+"      # GPU
    r"([\d.]+)\s+"   # QTY
    r"(\S+)\s+"      # CLOUD
    r"(\S+)\s+"      # INSTANCE_TYPE
    r"(\S+)\s+"      # DEVICE_MEM
    r"(\d+)\s+"      # vCPUs
    r"(\S+)\s+"      # HOST_MEM
    r"\$\s*([\d.]+)\s+"  # HOURLY_PRICE
    r"\$\s*([\d.]+)\s+"  # HOURLY_SPOT_PRICE
    r"(\S+)"         # REGION
)

rows = [m.groups() for m in pattern.finditer(data)]
columns = ["GPU","QTY","CLOUD","INSTANCE_TYPE","DEVICE_MEM","vCPUs","HOST_MEM","HOURLY_PRICE","HOURLY_SPOT_PRICE","REGION"]

df = pd.DataFrame(rows, columns=columns)

# Convertir tipos
df['QTY'] = df['QTY'].astype(float)
df['vCPUs'] = df['vCPUs'].astype(int)
df['HOURLY_PRICE'] = df['HOURLY_PRICE'].astype(float)
df['HOURLY_SPOT_PRICE'] = df['HOURLY_SPOT_PRICE'].astype(float)

df

,GPU,QTY,CLOUD,INSTANCE_TYPE,DEVICE_MEM,vCPUs,HOST_MEM,HOURLY_PRICE,HOURLY_SPOT_PRICE,REGION
0,RTX2000-Ada,1.0,RunPod,1x_RTX2000-Ada_SECURE,0GB,6,31GB,0.24,0.14,CA
1,RTX2000-Ada,2.0,RunPod,2x_RTX2000-Ada_SECURE,0GB,12,62GB,0.48,0.28,CA
2,RTX2000-Ada,3.0,RunPod,3x_RTX2000-Ada_SECURE,0GB,18,93GB,0.72,0.42,CA
3,RTX2000-Ada,4.0,RunPod,4x_RTX2000-Ada_SECURE,0GB,24,124GB,0.96,0.56,CA
4,RTX2000-Ada,5.0,RunPod,5x_RTX2000-Ada_SECURE,0GB,30,155GB,1.20,0.70,CA
5,RTX2000-Ada,6.0,RunPod,6x_RTX2000-Ada_SECURE,0GB,36,186GB,1.44,0.84,CA
6,RTX2000-Ada,7.0,RunPod,7x_RTX2000-Ada_SECURE,0GB,42,217GB,1.68,0.98,CA
7,RTX2000-Ada,8.0,RunPod,8x_RTX2000-Ada_SECURE,0GB,48,248GB,1.92,1.12,CA


In [3]:
import requests
import os

API_KEY = os.getenv("RUNPOD_API_KEY")

url = f"https://api.runpod.io/graphql?api_key={API_KEY}"

query = {
    "query": """
    query {
      gpuTypes(input: { id: "NVIDIA RTX 4000 Ada Generation" }) {
        id
        displayName
        lowestPrice(
          input: {
            secureCloud: true,
            gpuCount: 1,
            minVcpuCount: 2,
            minMemoryInGb: 8,
            minDisk: 0,
            dataCenterId: null,
            globalNetwork: false,
            compliance: null
          }
        ) {
          stockStatus
          availableGpuCounts
          maxUnreservedGpuCount
          uninterruptablePrice
          minimumBidPrice
        }
      }
    }
    """
}

headers = {
    "Content-Type": "application/json"
}

response = requests.post(url, json=query, headers=headers)

print(response.status_code)
print(response.json())

200
{'data': {'gpuTypes': [{'id': 'NVIDIA RTX 4000 Ada Generation', 'displayName': 'RTX 4000 Ada', 'lowestPrice': {'stockStatus': 'Low', 'availableGpuCounts': [1], 'maxUnreservedGpuCount': 1, 'uninterruptablePrice': 0.26, 'minimumBidPrice': 0.19}}]}}


In [ ]:
from vastai_sdk import VastAI

vast_sdk = VastAI(api_key='')

# Get all available GPU names
output = vast_sdk.search_offers(gpu_name="RTX_5090", rented=False, rentable=True, geolocation="US")
print(output)

[{'id': 30409089, 'ask_contract_id': 30409089, 'bundle_id': 2018009266, 'bundled_results': None, 'bw_nvlink': 0.0, 'compute_cap': 1200, 'cpu_arch': 'amd64', 'cpu_cores': 64, 'cpu_cores_effective': 64.0, 'cpu_ghz': 5.352, 'cpu_name': 'AMD Ryzen Threadripper PRO 7975WX 32-Cores', 'cpu_ram': 128280, 'credit_discount_max': 0.0, 'cuda_max_good': 13.0, 'direct_port_count': 99, 'disk_bw': 4460.0, 'disk_name': 'AMI Virtual', 'disk_space': 1309.0, 'dlperf': 398.01470754705707, 'dlperf_per_dphtotal': 595.8634379689793, 'dph_base': 0.6666666666666666, 'dph_total': 0.667962962962963, 'driver_version': '580.95.05', 'driver_vers': 580095005, 'duration': 1496348.28014493, 'end_date': 1775466433.0140142, 'external': None, 'flops_per_dphtotal': 322.117979484336, 'geolocation': 'Latvia, LV', 'geolocode': 1165216004, 'gpu_arch': 'nvidia', 'gpu_display_active': True, 'gpu_frac': 1.0, 'gpu_ids': [155078, 155079], 'gpu_lanes': 16, 'gpu_mem_bw': 1455.3, 'gpu_name': 'RTX 5090', 'gpu_ram': 32607, 'gpu_total_ra

In [41]:
import subprocess
import json

result = subprocess.run(["sky", "show-gpus", "--infra", "aws", "-a"], capture_output=True, text=True)
print(result.stdout)

⠋ Updating AWS catalog: aws/vms.csv (every 7 hours)
⠙ Updating AWS catalog: aws/vms.csv (every 7 hours)
⠹ Updating AWS catalog: aws/vms.csv (every 7 hours)
⠸ Updating AWS catalog: aws/vms.csv (every 7 hours)
⠼ Updating AWS catalog: aws/vms.csv (every 7 hours)
⠼ Updating AWS catalog: aws/vms.csv (every 7 hours)

COMMON_GPU  AVAILABLE_QUANTITIES       
A10G        1, 4, 8                    
A100        8                          
A100-80GB   8                          
B200        8                          
H100        1, 8                       
H200        8                          
L4          0.125, 0.25, 0.5, 1, 4, 8  
L40S        1, 4, 8                    
T4          1, 4, 8                    
V100        1, 4, 8                    
V100-32GB   8                          

OTHER_GPU        AVAILABLE_QUANTITIES  
B300             8                     
Gaudi HL-205     8                     
Inferentia       1, 4, 16              
Inferentia2      1, 6, 12              
RTXPRO

In [36]:
import re
import pandas as pd

text = result.stdout


# =========================
# HELPERS
# =========================

def parse_quantities(q_str):
    """Parsea cantidades (int + float) de forma robusta."""
    values = []
    for x in q_str.split(","):
        x = x.strip()
        try:
            val = float(x)
            if val.is_integer():
                val = int(val)
            values.append(val)
        except:
            continue
    return values


def parse_qty(x):
    """Parsea qty individual (puede ser float tipo 0.5)."""
    try:
        val = float(x)
        return int(val) if val.is_integer() else val
    except:
        return None


# =========================
# COMMON_GPU
# =========================

common_pattern = re.search(
    r"COMMON_GPU\s+AVAILABLE_QUANTITIES\s+(.*?)\n\n",
    text,
    re.DOTALL
)

common_rows = []
if common_pattern:
    for row in common_pattern.group(1).strip().split("\n"):
        m = re.match(r"(\S+)\s+(.+)", row.strip())
        if m:
            common_rows.append({
                "gpu": m.group(1),
                "quantities": parse_quantities(m.group(2))
            })

df_common = pd.DataFrame(common_rows)


# =========================
# OTHER_GPU
# =========================

other_pattern = re.search(
    r"OTHER_GPU\s+AVAILABLE_QUANTITIES\s+(.*?)\n\nGPU\s+QTY",
    text,
    re.DOTALL
)

other_rows = []
if other_pattern:
    for row in other_pattern.group(1).strip().split("\n"):
        m = re.match(r"(\S+)\s+(.+)", row.strip())
        if m:
            other_rows.append({
                "gpu": m.group(1),
                "quantities": parse_quantities(m.group(2))
            })

df_other = pd.DataFrame(other_rows)


# =========================
# PRICING
# =========================

pricing_blocks = re.findall(
    r"GPU\s+QTY\s+CLOUD.*?\n(.*?)(?=\nGPU\s+QTY|\Z)",
    text,
    re.DOTALL
)

rows = []

row_pattern = re.compile(
    r"(\S+)\s+"          # GPU
    r"([\d.]+)\s+"       # QTY
    r"(\S+)\s+"          # CLOUD
    r"(\S+)\s+"          # INSTANCE TYPE
    r"(\S+)\s+"          # DEVICE MEM
    r"(\d+)\s+"          # VCPUS
    r"(\S+)\s+"          # HOST MEM
    r"\$\s*([\d.]+)\s+"  # PRICE
    r"\$\s*([\d.]+)"     # SPOT PRICE
)

for block in pricing_blocks:
    for line in block.strip().split("\n"):
        m = row_pattern.match(line.strip())
        if m:
            rows.append({
                "gpu": m.group(1),
                "qty": parse_qty(m.group(2)),
                "cloud": m.group(3),
                "instance_type": m.group(4),
                "device_mem": m.group(5),
                "vcpus": int(m.group(6)),
                "host_mem": m.group(7),
                "price": float(m.group(8)),
                "spot_price": float(m.group(9)),
            })

df_pricing = pd.DataFrame(rows)


# =========================
# OPTIONAL: EXPLODE QUANTITIES
# =========================

df_common_expanded = df_common.explode("quantities")
df_other_expanded = df_other.explode("quantities")


# =========================
# OPTIONAL: NORMALIZE PRICE PER GPU
# =========================

def safe_divide(price, qty):
    try:
        return price / qty if qty else None
    except:
        return None

df_pricing["price_per_gpu"] = df_pricing.apply(
    lambda x: safe_divide(x["price"], x["qty"]),
    axis=1
)

df_pricing["spot_price_per_gpu"] = df_pricing.apply(
    lambda x: safe_divide(x["spot_price"], x["qty"]),
    axis=1
)


# =========================
# OUTPUT
# =========================

print("\n=== COMMON GPU ===")
print(df_common)

print("\n=== OTHER GPU ===")
print(df_other)

print("\n=== PRICING ===")
print(df_pricing.head())

print("\n=== COMMON EXPANDED ===")
print(df_common_expanded.head())


=== COMMON GPU ===
          gpu                   quantities
0        A10G                    [1, 4, 8]
1        A100                          [8]
2   A100-80GB                          [8]
3        B200                          [8]
4        H100                       [1, 8]
5        H200                          [8]
6          L4  [0.125, 0.25, 0.5, 1, 4, 8]
7        L40S                    [1, 4, 8]
8          T4                    [1, 4, 8]
9        V100                    [1, 4, 8]
10  V100-32GB                          [8]

=== OTHER GPU ===
           gpu    quantities
0         B300           [8]
1        Gaudi            []
2   Inferentia    [1, 4, 16]
3  Inferentia2    [1, 6, 12]
4   RTXPRO6000  [1, 2, 4, 8]
5       Radeon        [2, 4]
6          T4g        [1, 2]
7     Trainium       [1, 16]
8    Trainium2          [16]

=== PRICING ===
         gpu  qty cloud  instance_type device_mem  vcpus host_mem   price  \
0       A100  8.0   AWS   p4d.24xlarge       40GB     96   11

In [37]:
df_other

,gpu,quantities
0,B300,[8]
1,Gaudi,[]
2,Inferentia,"[1, 4, 16]"
3,Inferentia2,"[1, 6, 12]"
4,RTXPRO6000,"[1, 2, 4, 8]"
5,Radeon,"[2, 4]"
6,T4g,"[1, 2]"
7,Trainium,"[1, 16]"
8,Trainium2,[16]


In [30]:
import requests
import os

API_KEY = os.getenv("RUNPOD_API_KEY")
url = f"https://api.runpod.io/graphql?api_key={API_KEY}"

def run_query(query):
    response = requests.post(
        url,
        json={"query": query},
        headers={"Content-Type": "application/json"}
    )
    return response.json()["data"]

# -------- GPUs --------
gpus = run_query("""
query {
  gpuTypes {
    id
    displayName
  }
}
""")["gpuTypes"]

# -------- REGIONS --------
regions = run_query("""
query {
  dataCenters {
    id
    name
    location
  }
}
""")["dataCenters"]

print("GPUs:", len(gpus))
print("Regions:", len(regions))

GPUs: 43
Regions: 44


In [39]:
import boto3
import pandas as pd

def find_instance_globally(instance_type):
    # Cliente inicial para obtener la lista de todas las regiones activas
    ec2_global = boto3.client('ec2', region_name='us-east-1')
    regions = [r['RegionName'] for r in ec2_global.describe_regions()['Regions']]
    
    results = []

    print(f"Buscando {instance_type} en todas las regiones...")

    for region in regions:
        try:
            ec2_regional = boto3.client('ec2', region_name=region)
            
            # Consultamos si el tipo de instancia existe en esa región y en qué zonas
            response = ec2_regional.describe_instance_type_offerings(
                LocationType='availability-zone',
                Filters=[{'Name': 'instance-type', 'Values': [instance_type]}]
            )
            
            offerings = response['InstanceTypeOfferings']
            
            if offerings:
                for o in offerings:
                    results.append({
                        'Region': region,
                        'AvailabilityZone': o['Location'],
                        'InstanceType': o['InstanceType'],
                        'Status': 'Físicamente Disponible'
                    })
            else:
                # Si quieres registrar que NO está en la región, descomenta esto:
                # results.append({'Region': region, 'AvailabilityZone': 'N/A', 'Status': 'No ofrecida'})
                pass
                
        except Exception as e:
            print(f"No se pudo consultar la región {region}: {e}")

    return pd.DataFrame(results)

# --- EJECUCIÓN ---
# Cambia 'g5.2xlarge' por la GPU que necesites (ej. 'p4d.24xlarge', 'g4dn.xlarge')
mi_gpu = 'g5.2xlarge'
df_disponibilidad = find_instance_globally(mi_gpu)

# Mostrar resultados ordenados por Región
print(df_disponibilidad.sort_values(by='Region'))


Buscando g5.2xlarge en todas las regiones...
            Region AvailabilityZone InstanceType                  Status
13  ap-northeast-1  ap-northeast-1c   g5.2xlarge  Físicamente Disponible
12  ap-northeast-1  ap-northeast-1a   g5.2xlarge  Físicamente Disponible
10  ap-northeast-2  ap-northeast-2d   g5.2xlarge  Físicamente Disponible
11  ap-northeast-2  ap-northeast-2a   g5.2xlarge  Físicamente Disponible
9   ap-northeast-2  ap-northeast-2c   g5.2xlarge  Físicamente Disponible
0       ap-south-1      ap-south-1a   g5.2xlarge  Físicamente Disponible
1       ap-south-1      ap-south-1b   g5.2xlarge  Físicamente Disponible
19  ap-southeast-2  ap-southeast-2a   g5.2xlarge  Físicamente Disponible
18  ap-southeast-2  ap-southeast-2c   g5.2xlarge  Físicamente Disponible
14    ca-central-1    ca-central-1b   g5.2xlarge  Físicamente Disponible
15    ca-central-1    ca-central-1a   g5.2xlarge  Físicamente Disponible
22    eu-central-1    eu-central-1b   g5.2xlarge  Físicamente Disponible
21    

In [49]:
import requests
import os
import pandas as pd

API_KEY = os.getenv("RUNPOD_API_KEY")
url = f"https://api.runpod.io/graphql?api_key={API_KEY}"

def run_query(query):
    response = requests.post(
        url,
        json={"query": query},
        headers={"Content-Type": "application/json"}
    )
    return response.json()["data"]

# -------- INPUT --------
GPU_ID = "NVIDIA RTX 2000 Ada Generation"

print(f"Buscando {GPU_ID} en todas las regiones...")

# -------- REGIONS --------
regions = run_query("""
query {
  dataCenters {
    id
    name
    location
  }
}
""")["dataCenters"]

results = []

# -------- LOOP POR REGION --------
for region in regions:
    region_id = region["id"]
    region_name = region["name"]

    query = f"""
    query {{
      gpuTypes(input: {{ id: "{GPU_ID}" }}) {{
        id
        displayName
        lowestPrice(
          input: {{
            secureCloud: true,
            gpuCount: 1,
            minVcpuCount: 2,
            minMemoryInGb: 8,
            minDisk: 0,
            dataCenterId: "{region_id}",
            globalNetwork: false,
            compliance: null
          }}
        ) {{
          stockStatus
          availableGpuCounts
          maxUnreservedGpuCount
        }}
      }}
    }}
    """

    try:
        data = run_query(query)["gpuTypes"][0]["lowestPrice"]

        status = data["stockStatus"]

        # 🚨 FILTRO AQUÍ
        if status is None:
            continue

        if status == "AVAILABLE":
            status_readable = "Físicamente Disponible"
        elif status == "OUT_OF_STOCK":
            status_readable = "Sin stock"
        else:
            status_readable = status

        results.append({
            "Region": region_name,
            "GPU": GPU_ID,
            "AvailableCounts": data["availableGpuCounts"],
            "MaxAvailable": data["maxUnreservedGpuCount"],
            "Status": status_readable
        })

    except Exception as e:
        # Algunas regiones no tienen ese GPU
        results.append({
            "Region": region_name,
            "GPU": GPU_ID,
            "AvailableCounts": None,
            "MaxAvailable": 0,
            "Status": "No disponible"
        })

# -------- DATAFRAME --------
df = pd.DataFrame(results)

# opcional: ordenar por disponibilidad
df = df.sort_values(by="MaxAvailable", ascending=False)

print(df)

Buscando NVIDIA RTX 2000 Ada Generation en todas las regiones...
     Region                             GPU AvailableCounts  MaxAvailable  \
0   EU-RO-1  NVIDIA RTX 2000 Ada Generation          [1, 2]             2   
1  EUR-IS-1  NVIDIA RTX 2000 Ada Generation             [1]             1   

  Status  
0    Low  
1    Low  


In [48]:
df

,Region,RegionID,GPU,AvailableCounts,MaxAvailable,Status
9,EU-RO-1,EU-RO-1,NVIDIA RTX 2000 Ada Generation,[1],1,Low
12,EUR-IS-1,EUR-IS-1,NVIDIA RTX 2000 Ada Generation,[1],1,Low
2,CA-MTL-2,CA-MTL-2,NVIDIA RTX 2000 Ada Generation,[],0,None
0,AP-JP-1,AP-JP-1,NVIDIA RTX 2000 Ada Generation,[],0,None
3,CA-MTL-3,CA-MTL-3,NVIDIA RTX 2000 Ada Generation,[],0,None
4,CA-MTL-4,CA-MTL-4,NVIDIA RTX 2000 Ada Generation,[],0,None
6,EU-DK-1,EU-DK-1,NVIDIA RTX 2000 Ada Generation,[],0,None
5,EU-CZ-1,EU-CZ-1,NVIDIA RTX 2000 Ada Generation,[],0,None
7,EU-FR-1,EU-FR-1,NVIDIA RTX 2000 Ada Generation,[],0,None
8,EU-NL-1,EU-NL-1,NVIDIA RTX 2000 Ada Generation,[],0,None


In [62]:
import pandas as pd
from vastai_sdk import VastAI
import os

vast_sdk = VastAI(api_key=os.getenv("VAST_AI_KEY"))

def get_vast_exact(gpu_id="RTX 4090"):
    # IMPORTANTE: Cambiamos ~ por = y ponemos el ID entre comillas dobles
    # Esto obliga a que el nombre sea exactamente el que pides.
    query = f'gpu_name = "{gpu_id}" rentable = True'
    
    try:
        offers = vast_sdk.search_offers(query=query)
        if not offers:
            print(f"No se encontraron ofertas exactas para: {gpu_id}")
            return None

        df = pd.DataFrame(offers)

        # Mapeo de columnas (mantenemos la lógica flexible por seguridad)
        price_col = next((c for c in ['dph_total', 'rentable_price', 'price'] if c in df.columns), None)
        loc_col = next((c for c in ['geolocation', 'geolocation_caption', 'location'] if c in df.columns), None)
        gpu_col = next((c for c in ['num_gpus', 'gpu_count'] if c in df.columns), None)

        if not price_col or not gpu_col or not loc_col:
            print("Error: No se identificaron las columnas necesarias.")
            return None

        # Asegurar tipos numéricos
        df[gpu_col] = pd.to_numeric(df[gpu_col], errors='coerce').fillna(0)
        df[price_col] = pd.to_numeric(df[price_col], errors='coerce').fillna(0)

        # Agrupación por Región
        summary = df.groupby(loc_col).agg({
            'id': 'count',
            gpu_col: 'sum',
            price_col: 'min'
        }).rename(columns={
            'id': 'Hosts',
            gpu_col: 'Total_GPUs',
            price_col: 'Precio_Min_Hr'
        })

        return summary.sort_values(by='Total_GPUs', ascending=False)

    except Exception as e:
        print(f"Error detallado: {e}")
        return None

# Ejecución con ID exacto
# Nota: "RTX 4090" es el standard de Vast, pero asegúrate de que sea el ID correcto
df_vast = get_vast_exact("RTX 4090")

if df_vast is not None:
    print(f"Resultados exactos para RTX 4090:")
    print(df_vast)


Resultados exactos para RTX 4090:
                      Hosts  Total_GPUs  Precio_Min_Hr
geolocation                                           
Iceland, IS               8          11       0.348056
Texas, US                 8          11       0.375185
Japan, JP                 3          10       1.000926
The Netherlands, NL       6           8       0.308056
Spain, ES                 5           6       0.277852
Hungary, HU               5           6       0.301852
United Kingdom, GB        5           6       0.375472
France, FR                4           5       0.268056
Bulgaria, BG              2           4       0.534722
Washington, US            3           4       0.547185
Illinois, US              2           3       0.354583
Quebec, CA                2           3       0.308056
Utah, US                  2           3       0.427593
Sweden, SE                2           2       0.328981
Nebraska, US              1           2       0.720463
British Columbia, CA      1    

In [63]:
import pandas as pd
from vastai_sdk import VastAI
import os

vast_sdk = VastAI(api_key=os.getenv("VAST_AI_KEY"))

def get_vast_gpu_catalog():
    try:
        # Consultamos todas las ofertas de alquiler disponibles en el marketplace
        # Sin filtros de nombre para traer todo el inventario
        offers = vast_sdk.search_offers(query="rentable = True")
        
        if not offers:
            print("No se pudieron recuperar ofertas.")
            return None

        df = pd.DataFrame(offers)

        # Buscamos la columna de nombre de GPU (usualmente 'gpu_name')
        gpu_col = next((c for c in ['gpu_name', 'model', 'gpu_model'] if c in df.columns), None)
        
        if not gpu_col:
            print("No se encontró la columna de nombre de GPU.")
            return None

        # Obtenemos la lista única de modelos, ordenada alfabéticamente
        gpu_list = sorted(df[gpu_col].unique())
        
        # Opcional: Contar cuántas unidades hay de cada una en todo el marketplace
        catalog_summary = df.groupby(gpu_col).agg({
            'id': 'count',
            'num_gpus': 'sum'
        }).rename(columns={'id': 'Total_Hosts', 'num_gpus': 'Total_Units'}).sort_values('Total_Units', ascending=False)

        return catalog_summary

    except Exception as e:
        print(f"Error al obtener el catálogo: {e}")
        return None

# Ejecución
catalogo_gpus = get_vast_gpu_catalog()

if catalogo_gpus is not None:
    print("Catálogo de GPUs disponibles en Vast.ai (IDs exactos):")
    print(catalogo_gpus)
    
    # Si solo quieres ver los nombres para copiar y pegar:
    print("\nLista de IDs para búsqueda exacta:")
    print(list(catalogo_gpus.index))


Catálogo de GPUs disponibles en Vast.ai (IDs exactos):
                 Total_Hosts  Total_Units
gpu_name                                 
RTX 5090                   7           22
H200                       5           16
RTX PRO 6000 S             5           10
RTX 4070S Ti               3            7
RTX 4070 Ti                3            7
RTX 5080                   3            7
RTX 4090                   4            7
RTX PRO 4000               4            6
H100 NVL                   3            5
RTX 5060 Ti                3            5
RTX PRO 5000               3            4
RTX PRO 6000 WS            3            4
RTX 6000Ada                2            3
RTX 5070 Ti                2            3
RTX 5070                   2            3
RTX 4070                   2            3
H200 NVL                   2            2
RTX 3090                   1            1
L40                        1            1
B200                       1            1
H100 SXM             

In [65]:
import pandas as pd
from vastai_sdk import VastAI
import os

vast_sdk = VastAI(api_key=os.getenv("VAST_AI_KEY"))

def get_vast_human_readable(gpu_id="RTX 4090"):
    query = f'gpu_name = "{gpu_id}" rentable = True'
    
    try:
        offers = vast_sdk.search_offers(query=query)
        if not offers:
            print(f"No se encontraron ofertas para: {gpu_id}")
            return None

        df = pd.DataFrame(offers)

        # 1. Conversiones de Unidades
        # Red: Mbps -> MB/s (Megabytes por segundo)
        df['MBs_Down'] = (pd.to_numeric(df['inet_down']) / 8).round(1)
        df['MBs_Up'] = (pd.to_numeric(df['inet_up']) / 8).round(1)
        
        # Costos: $/GB -> $/TB
        df['Egress_$/TB'] = (pd.to_numeric(df['inet_up_cost']) * 1024).round(2)
        df['Ingress_$/TB'] = (pd.to_numeric(df['inet_down_cost']) * 1024).round(2)
        
        # VRAM: MB -> GB
        df['VRAM_GB'] = (pd.to_numeric(df['gpu_ram']) / 1024).astype(int)
        
        # Confiabilidad: 0.99 -> 99%
        df['Reliability_%'] = (pd.to_numeric(df['reliability2']) * 100).round(1)

        # 2. Selección de columnas con nombres entendibles
        cols_map = {
            'id': 'Offer_ID',
            'geolocation': 'Region',
            'num_gpus': 'GPUs',
            'VRAM_GB': 'VRAM_GB',
            'dph_total': 'Price_$/Hr',
            'MBs_Down': 'Descarga_MB/s',
            'MBs_Up': 'Subida_MB/s',
            'Egress_$/TB': 'Costo_Egress_$/TB',
            'Ingress_$/TB': 'Costo_Ingress_$/TB',
            'Reliability_%': 'Reliability_%'
        }

        df_final = df[list(cols_map.keys())].rename(columns=cols_map)

        # Ordenar por el más barato
        return df_final.sort_values(by='Price_$/Hr', ascending=True)

    except Exception as e:
        print(f"Error detallado: {e}")
        return None

# Ejecución
df_4090 = get_vast_human_readable("RTX 4090")

if df_4090 is not None:
    print(f"Listado detallado de RTX 4090 (Unidades legibles):")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    # Mostramos los primeros 30 resultados para no saturar
    print(df_4090.head(30).to_string(index=False))


Listado detallado de RTX 4090 (Unidades legibles):
 Offer_ID               Region  GPUs  VRAM_GB  Price_$/Hr  Descarga_MB/s  Subida_MB/s  Costo_Egress_$/TB  Costo_Ingress_$/TB  Reliability_%
 31239993       California, US     1       23    0.242315          194.8         65.1               4.00                2.67           99.0
 28758521           Quebec, CA     1       23    0.254546          110.7        111.0               0.00                0.00           97.3
 32642705           France, FR     1       23    0.268056           96.8         96.9               4.00                2.67           98.3
 31129843           France, FR     1       23    0.268056          107.7         92.9               4.00                2.67           99.2
 33135922           Norway, NO     1       23    0.268519          101.5        100.8              13.65               13.65           99.7
 30950585  The Netherlands, NL     1       23    0.281389          135.0        168.9               1.33     